## 0. Setup

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath('.'))   # folder containing dgc_pipeline/

from dgc_pipeline import (Backend, OperatorSequence, FusedOperatorBuilder,
                          DGCOperatorBuilder, TargetBuilder, TrajectoryBuilder,
                          BOnlyScorer, DynamicsScorer, ControlBestDayScorer,
                          InterleavedFusion, NormanBBuilder,
                          make_response_sum, tikhonov_deconvolve,
                          fusion_vs_averaging, forcing_svd, forcing_geometry)
from dgc_pipeline import io as dio

DATA = dict(
    expr      = 'data/ExprMatrix.var.genes.h5ad',
    cell_days = 'data/cell_days.txt',
    cell_sets = 'major_cell_sets.gmt',
    tmap_dir  = 'tmaps',
    b_cache   = 'b_cache/B_binding_07a8161f4b109116.npz',
    traj_cache= 'traj_cache.npz',
    norman    = 'data/fibroblast_CRISPRa_custom_mean.h5ad',   # or _mean_pop.h5ad
    norman_out= 'b_norman.npz',
)
USE_GPU = True
IPSC_SET = 'iPSC'

## 1. Backend

In [ ]:
be = (Backend.gpu(require=True) if USE_GPU else Backend.cpu()).activate()
print('on_gpu:', be.on_gpu)

## 2. Load expression, cell days, transport maps

In [ ]:
A = dio.load_expression(DATA['expr'])
genes = np.asarray(A.var_names)
cell_days = dio.load_cell_days(DATA['cell_days'])
tmap_chain = dio.discover_tmaps(DATA['tmap_dir'])
print(f'genes={len(genes)}  tmaps={len(tmap_chain)}  cells-with-days={len(cell_days)}')

## 3. Trajectories — run EITHER 3A (load) OR 3B (build + save)

In [ ]:
# if os.path.exists(DATA['traj_cache']):
#     Z = TrajectoryBuilder.load(DATA['traj_cache'])
#     traj_cache, xbar, panel = Z['traj_c'], Z['xbar'], Z['panel']
#     seed_ids = list(Z['seed_ids']) if 'seed_ids' in Z else None
#     fate     = Z['fate'] if 'fate' in Z else None
#     L, K, G = traj_cache.shape; n_steps = K - 1
#     print(f'loaded: L={L} K={K} G={G}; panel={len(panel)}')
# else:
#     print('no cache yet -- run 3B')

In [ ]:
# 3b
tbuild = TrajectoryBuilder(tmap_chain, A, genes)
traj, seed_ids = tbuild.build()
traj_c, xbar   = TrajectoryBuilder.center(traj)
panel          = TrajectoryBuilder.panel(traj, n_top=None)
tb_t = TargetBuilder(A, genes, DATA['cell_sets'], ipsc_key_substr=IPSC_SET)
ipsc_rows = tb_t._resolve_rows()
ipsc_cell_ids = np.asarray(A.obs_names)[ipsc_rows] if len(ipsc_rows) else []
fate = tbuild.seed_fate(seed_ids, ipsc_cell_ids) if len(ipsc_cell_ids) else None
L, K, G = traj_c.shape; n_steps = K - 1
TrajectoryBuilder.save(DATA['traj_cache'], traj_c, xbar, panel,
                       seed_ids=seed_ids, fate=fate)
print(f'built + saved: L={L} K={K} G={G}; panel={len(panel)}')

In [ ]:


def enforce_uniform_resolution(traj):
    """
    Removes irregular 6-hour time steps (Day 8.25 and 8.75) from the dataset
    to strictly enforce a 0.5-day resolution, matching the discrete DMD model assumptions.
    
    traj: shape (n_samples, n_timepoints, n_features) - expects 39 timepoints
    """
    # 1. Exact WOT Time Schedule (39 points)
    wot_times = np.array([
        0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 
        5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.25, 8.5, 8.75, 
        9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 
        14.0, 14.5, 15.0, 15.5, 16.0, 16.5, 17.0, 17.5, 18.0
    ])
    
    # Verify trajectory matches the expected 39 timepoints
    if traj.shape[1] != len(wot_times):
        raise ValueError(f"Expected trajectory with {len(wot_times)} timepoints, got {traj.shape[1]}")
        
    # Identify indices where the time step is exactly on a 0.5-day increment
    valid_indices = np.where(wot_times % 0.5 == 0)[0]
    
    # Slice the trajectory and times to keep only uniform steps
    traj_uniform = traj[:, valid_indices, :]
    times_uniform = wot_times[valid_indices]
    
    print(f"Original shape: {traj.shape} (Includes Days 8.25, 8.75)")
    print(f"Uniform shape:  {traj_uniform.shape} (Strict 0.5-day steps)")
    
    return traj_uniform, times_uniform
traj_u=enforce_uniform_resolution(traj_cache)[0]
traj_raw = traj_u + xbar[np.newaxis, np.newaxis, :]

# traj_c shape: (trials, times, features)
mean_initial = traj_raw[:, 0, :].mean(axis=0)   # shape: (features,)
traj_c= traj_raw - mean_initial[np.newaxis, np.newaxis, :] # This centers using the initial unperturbed state

## Predictors

In [ ]:
# Utilities
import numpy as np

def test(x_ts, y_true, f, t, var=1):
    """
    Test any given predictor on data at a given time, calculate pooled R^2,
    per-gene averaged R^2, and mean absolute error.
    """
    predictions = f(x_ts, t)
    
    # --- Pooled R^2 (current behavior) ---
    global_mse = np.mean((y_true - predictions) ** 2)
    global_var = np.mean((y_true - np.mean(y_true, axis=0)) ** 2)
    r2_pooled = 1 - (global_mse / global_var)
    
    # --- Per-gene R^2, averaged over genes ---
    ss_res = np.sum((y_true - predictions) ** 2, axis=0)   # per gene
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0)) ** 2, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        gene_r2 = 1 - ss_res / ss_tot
    # Exclude genes with zero variance (where ss_tot == 0)
    valid = np.isfinite(gene_r2) & (ss_tot > 1e-12)
    r2_per_gene_avg = np.mean(gene_r2[valid]) if np.any(valid) else np.nan
    
    # --- Mean absolute error ---
    mae = np.mean(np.abs(y_true - predictions))
    
    return r2_pooled, r2_per_gene_avg, mae, predictions

import numpy as np

def test_model(x, f,var,times):
    """
    Test a model on a dataset, calculate r^2 and mean absolute error for all timepoints
    """
    results = {}
    # Data is an array of shape (n_samples, n_timepoints, n_features)
    # Predictor f takes x(t) and returns predictions for x(t+1), where x is a matrix of all n samples of x at time t, and so on.
    
    # We iterate up to n_timepoints - 1 because we need a t+1 to compare against
    x_t = x[:, 0, :]         # matrix of all n samples at time 0
    for t in range(x.shape[1]-1 ):       # matrix of all n samples at time t
        x_next = x[:, t+1, :]  # true values for all n samples at time t+1
        
        # Test predictions for time t+1 using inputs from time t
        r2_pooled, r2_individual, mae, predictions = test(x_t, x_next, f, t, var)
        x_t = predictions
        results[t] = {'r2_pooled': r2_pooled, 'r2_individual': r2_individual, 'mae': mae}
        if t in times:
            print(f"Time {t}: r^2 pooled = {r2_pooled:.4f},  mae = {mae:.4f}")

    return results

def test_predictor(x, predictor_func, times):
    """
    Test a predictor function on the data at specified timepoints.
    predictor_func is a function that takes (x, t) and returns predictions for x(t+1).
    """
    results = test_model(x, predictor_func,None,times)
    

def differencing_data(data):
    """
    Do pair differencing of data to cancel out the affine term.
    data is in shape (n_samples, n_timepoints, n_features)
    """
    # np.roll shifts the samples down by 1. 
    # This guarantees every sample is paired with a different sample (no self-pairing).
    return data - np.roll(data, shift=1, axis=0)
def tdifferencing_data(data):
    """ Perform temporal differencing"""
    return data[:,1:,:]-data[:,:-1,:]

In [ ]:
data = traj_c
data2 = data[:, 16:, :] # DOX OFF data for control inference

# Models

In [ ]:
# Full model method
import numpy as np

def full_model(fused_op, data):
    A = fused_op.as_matrix(len(panel))
    
    # Calculate the mean across all samples for each timepoint
    # Shape of mean_data: (n_timepoints, n_features)
    mean_data = np.mean(data, axis=0) 
    
    # Calculate b for every timestep t.
    # mean_data[:-1].T shapes the data to (n_features, n_timepoints-1) so A can multiply it.
    # We transpose the result back to (n_timepoints-1, n_features) to subtract it from mean_data[1:].
    b = mean_data[1:] - (A @ mean_data[:-1].T).T
    
    # b is an array of shape (n_timepoints-1, n_features).
    # b[t] grabs the corresponding (n_features,) affine term for that timestep.
    return lambda x, t: (A @ x.T).T + b[t]#Note, x is a matrix of shape (G, N) where G is the number of genes and N is the number of samples


In [ ]:
#Stacked dmd method
import numpy as np

def stacked_DMD(x,y, diff = True,ridge=0.05,rank_p = 0.95):
    orig_data = x 
    if diff:
        diff_x = differencing_data(x)  # Lineage pair differencing: yt = xt - xt'
        diff_y = differencing_data(y)
    else:
        diff_x = x
        diff_y = y
    # 1. X is the current state
    X_T = diff_x.reshape(-1, diff_x.shape[2])  
    
    # Predict the NEXT state (Standard DMD), since we are using M = A
    Y_T_next = diff_y.reshape(-1, diff_y.shape[2]) 
    
    # 2. Compute covariance matrix C = X @ X^T -> (F, F)
    C = X_T.T @ X_T  
    
    # 3. Eigendecomposition to get U and S
    eigvals, eigvecs = np.linalg.eigh(C)
    
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    U = eigvecs[:, idx]
    
    valid = eigvals > 1e-10
    eigvals = eigvals[valid]
    U = U[:, valid]
    
    S = np.sqrt(eigvals)
    
    # 4. Rank truncation to capture 95% of variance
    variance_explained = np.cumsum(S**2) / np.sum(S**2)
    r = np.argmax(variance_explained >= rank_p) + 1
    
    Ur = U[:, :r]
    Sr = S[:r]
    
    # 5. Compute the operator A using the next state data
    YX_T_next = Y_T_next.T @ X_T
    
    # APPLY RIDGE REGULARIZATION HERE: 1 / (S^2 + ridge)
    A = YX_T_next @ Ur @ np.diag(1.0 / (Sr**2 + ridge)) @ Ur.T
    
    # 6. Construct the transition operator
    M = A
    
    # 7. Recover the affine term
    mean_x = np.mean(orig_data, axis=0) 
    mean_y = np.mean(y,axis=0)
    b = mean_y - (M @ mean_x.T).T
    
    return lambda x, t: (x @ M.T + b[t]), A

In [ ]:
times=[0,1,2,4,8,16,32]
# Returns a single float
var = data[:, 0, :].var(axis=0).sum()
dmd_model,dmd_A = stacked_DMD(data[:,:-1,:],data[:,1:,:])
dmd_a = lambda x,t: x@dmd_A.T

In [ ]:
fused_op, info = FusedOperatorBuilder(be).build(
    data, ridge=0.0, eps_fuse=0.05, gene_panel=panel,
    pair_lineages=True, pair_seed=0,
    explained_variance_threshold=0.9,   # 90% energy threshold per pair
    n_steps_hint=n_steps, stack=1)
    
print(f"\nrho(M_12)={info['rho']:.4f}  rank(D_12)={info['fused_rank']}")
fused_Model = full_model(fused_op, data)
fused_A = fused_op.as_matrix(len(panel))
fused_a = lambda x,t: x@fused_A.T
test_predictor(data, fused_Model, range(37))

## Here we infer control from autonomous conversion


In [ ]:
import numpy as np
from itertools import combinations_with_replacement
from sklearn.decomposition import PCA


class SparseNonlinearResidual:
    """
    Sparse polynomial-in-geometric-mean model of the nonlinear residual
    Y = f(X), fitted in a low-rank PCA coordinate system.

    Library term for exponent tuple idx of degree d:
        classic   : prod_m x_{idx_m}
        geometric : sign(prod_m x_{idx_m}) * (prod_m |x_{idx_m}|)^(1/d)

    Geometric-mean terms grow linearly rather than as x^d, so higher-degree
    terms no longer blow up at large x. This improves conditioning and
    stability of the fitted vector field.

    Parameters
    ----------
    rank_x, rank_y : int
        PCA ranks retained for X and Y.
    degree : int
        Maximum degree of the library.
    prune_threshold : float
        Relative term-contribution threshold.
    max_iter : int
        Max prune / refit iterations.
    ridge : float
        Ridge added to normal equations.
    geometric_mean : bool
        If True (default), use signed geometric means for degree >= 2 terms.
        If False, fall back to classic polynomial products.
    intercept : bool
        If True (default), include a constant term in the library and never
        prune it. If False, no constant term is included and all terms are
        subject to pruning. Note that because Y is PCA-centered before the
        fit, the intercept in PCA space is usually ~0; turning it off is
        mainly useful for diagnostics or for non-centered residuals.
    verbose : bool
        If True, print R^2 and active-term count each iteration.
    """

    def __init__(self, rank_x=10, rank_y=10, degree=2,
                 prune_threshold=1e-3, max_iter=20, ridge=1e-8,
                 geometric_mean=True, pure_squares=True,intercept=True, verbose=True):
        self.rank_x = rank_x
        self.rank_y = rank_y
        self.degree = degree
        self.prune_threshold = prune_threshold
        self.max_iter = max_iter
        self.ridge = ridge
        self.geometric_mean = geometric_mean
        self.intercept = intercept
        self.verbose = verbose
        self.pure_squares = pure_squares

    # ------------------------------------------------------------------
    # Library
    # ------------------------------------------------------------------
    @staticmethod
    def _term_powers(d, degree, intercept=True):
        powers = [()] if intercept else []
        for deg in range(1, degree + 1):
            for idx in combinations_with_replacement(range(d), deg):
                powers.append(idx)
        return powers

    @classmethod
    def _build_library(cls, Z, powers, geometric=True, pure_squares=True, tiny=1e-30):
        """
        For d == 0: constant 1.
        For d == 1: the value itself.
        For d >= 2:
            pure power (all indices equal) and pure_squares=True -> raw product
                e.g. (3,3) -> x_3 * x_3 = x_3^2
            otherwise, geometric=True  -> signed geometric mean
            otherwise                  -> raw product
        """
        n = Z.shape[0]
        cols = []
        for idx in powers:
            d = len(idx)
            if d == 0:
                cols.append(np.ones(n))
                continue
            vals = Z[:, idx]                                    # (n, d)
            if d == 1 or not geometric:
                cols.append(np.prod(vals, axis=1))
                continue
            if pure_squares and len(set(idx)) == 1:
                # genuine x_i^d, kept as a raw product
                cols.append(np.prod(vals, axis=1))
                continue
            # signed geometric mean for mixed / higher interactions
            abs_vals = np.abs(vals)
            log_abs  = np.log(abs_vals + tiny)
            geo_abs  = np.exp(log_abs.mean(axis=1))
            signs    = np.prod(np.sign(vals), axis=1)
            cols.append(signs * geo_abs)
        return np.stack(cols, axis=1)
    # ------------------------------------------------------------------
    # R^2
    # ------------------------------------------------------------------
    @staticmethod
    def _r2(Y_true, Y_pred):
        yt = Y_true.reshape(-1)
        yp = Y_pred.reshape(-1)
        ss_res = float(np.sum((yt - yp) ** 2))
        ss_tot = float(np.sum((yt - yt.mean()) ** 2)) + 1e-12
        return 1.0 - ss_res / ss_tot

    # ------------------------------------------------------------------
    # Fit
    # ------------------------------------------------------------------
    def fit(self, X, Y):
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        if X.shape != Y.shape:
            raise ValueError(
                f"X and Y must have the same shape, got {X.shape} and {Y.shape}"
            )

        self.in_shape_ = X.shape
        D = X.shape[-1]
        Xf = X.reshape(-1, D)
        Yf = Y.reshape(-1, D)

        rx = min(self.rank_x, D, Xf.shape[0])
        ry = min(self.rank_y, D, Yf.shape[0])

        self.pca_x_ = PCA(n_components=rx, svd_solver="full").fit(Xf)
        self.pca_y_ = PCA(n_components=ry, svd_solver="full").fit(Yf)

        Zx = self.pca_x_.transform(Xf)
        Zy = self.pca_y_.transform(Yf)

        self.powers_ = self._term_powers(rx, self.degree,
                                         intercept=self.intercept)
        Theta = self._build_library(Zx, self.powers_,
                                    geometric=self.geometric_mean, pure_squares=self.pure_squares)
        p = Theta.shape[1]

        if p == 0:
            raise ValueError(
                "Empty library: degree=0 with intercept=False. "
                "Nothing to fit."
            )

        Zy_rms = np.sqrt(np.mean(Zy ** 2)) + 1e-12
        n, ry_ = Zy.shape

        # index of the constant term in powers_, or None if no intercept
        self.intercept_idx_ = 0 if self.intercept else None

        active = np.ones(p, dtype=bool)
        Xi = np.zeros((p, ry_))

        if self.verbose:
            print(f"[SparseNonlinearResidual] library p = {p}, "
                  f"rank_x = {rx}, rank_y = {ry_}, degree = {self.degree}, "
                  f"geometric_mean = {self.geometric_mean}, "
                  f"intercept = {self.intercept}")

        for it in range(self.max_iter):
            Th = Theta[:, active]
            G = Th.T @ Th + self.ridge * np.eye(Th.shape[1])
            Xi_active = np.linalg.solve(G, Th.T @ Zy)

            Xi = np.zeros((p, ry_))
            Xi[active] = Xi_active

            Y_pred = self.pca_y_.inverse_transform(Theta @ Xi).reshape(Y.shape)
            r2 = self._r2(Y, Y_pred)

            if self.verbose:
                print(f"  iter {it:02d} | active terms = {active.sum():4d} | "
                      f"train R^2 = {r2:.6f}")

            theta_sq = (Theta ** 2).sum(axis=0)
            xi_sq = (Xi ** 2).sum(axis=1)
            term_rms = np.sqrt(theta_sq * xi_sq / (n * ry_))
            rel = term_rms / Zy_rms

            new_active = rel >= self.prune_threshold
            # only force-keep the constant if it exists
            if self.intercept_idx_ is not None:
                new_active[self.intercept_idx_] = True

            if np.array_equal(new_active, active):
                break
            active = new_active

        self.active_ = active
        self.Xi_ = Xi
        self.n_terms_ = int(active.sum())

        Y_pred = self.pca_y_.inverse_transform(Theta @ Xi).reshape(Y.shape)
        self.r2_ = self._r2(Y, Y_pred)

        if self.verbose:
            print(f"[SparseNonlinearResidual] done. "
                  f"active terms = {self.n_terms_}, train R^2 = {self.r2_:.6f}")
        return self

    # ------------------------------------------------------------------
    # Predict / score
    # ------------------------------------------------------------------
    def __call__(self, X):
        X = np.asarray(X, dtype=float)
        in_shape = X.shape
        D = in_shape[-1]
        Xf = X.reshape(-1, D)

        Zx = self.pca_x_.transform(Xf)
        Theta = self._build_library(Zx, self.powers_,
                                    geometric=self.geometric_mean, pure_squares=self.pure_squares)
        Zy_hat = Theta[:, self.active_] @ self.Xi_[self.active_]
        Yf_hat = self.pca_y_.inverse_transform(Zy_hat)
        return Yf_hat.reshape(in_shape)

    def score(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        return self._r2(Y, self(X))

    # ------------------------------------------------------------------
    # Introspection
    # ------------------------------------------------------------------
    def active_terms(self):
        return [pw for pw, a in zip(self.powers_, self.active_) if a]
def compute_bk(data, M, average=False):
    """
    data : (trials, times, features)
    M    : (features, features)
    """
    if data.ndim != 3:
        raise ValueError("data must be 3D")
    n_trials, n_times, n_features = data.shape
    if M.shape != (n_features, n_features):
        raise ValueError(f"M must have shape ({n_features}, {n_features})")

    # Residual for each trial and time step
    drift = data[:, 1:, :] - data[:, :-1, :] @ M.T

    if average:
        return drift.mean(axis=0)
    return drift
b_k_all = compute_bk(data,fused_A)

In [ ]:
extra = 0
rank = 2
deg = 2
model = SparseNonlinearResidual(
    rank_x=rank, rank_y=rank, degree=deg,
    prune_threshold=0.1, max_iter=20,
    geometric_mean=False,intercept=True,pure_squares=True).fit(data2[:,extra:-1,:], b_k_all[:,16+extra:,:])
model_poly = lambda x, t: model(x)+x@fused_A.T # Construct polynomial model

In [ ]:
control = data[:,1:,:]-(model_poly(data[:,:-1,:],0))
mean_c = control.mean(axis=0)